## Setup

In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from csv import QUOTE_NONE
from typing import Dict, List
from collections import defaultdict



# Load

In [4]:
LOCAL_DATA_ROOT = Path(os.environ.get("MIND_DATA_ROOT", "../smallDataset")).expanduser().resolve()

train_dir = LOCAL_DATA_ROOT / f"MINDsmall_train"
valid_dir = LOCAL_DATA_ROOT / f"MINDsmall_dev"

train_news_path      = str(train_dir / "news.tsv")
train_behaviors_path = str(train_dir / "behaviors.tsv")
valid_news_path      = str(valid_dir / "news.tsv")
valid_behaviors_path = str(valid_dir / "behaviors.tsv")

NEWS_COLUMNS = [
    "nid",
    "vertical",
    "subvertical",
    "title",
    "abstract",
    "url",
    "title_entities",
    "abstract_entities",
]
BEHAVIOR_COLUMNS = [
    "impression_id",
    "user_id",
    "time",
    "history",
    "impressions",
]

def load_tsv(path_str: str, columns):
    return pd.read_table(
        path_str,
        header=None,
        names=columns,
        sep='\t',
        quoting=QUOTE_NONE,
        dtype=object,
        na_filter=False,
    )

train_news_df = load_tsv(train_news_path, NEWS_COLUMNS)
valid_news_df = load_tsv(valid_news_path, NEWS_COLUMNS)
train_behaviors_df = load_tsv(train_behaviors_path, BEHAVIOR_COLUMNS)
valid_behaviors_df = load_tsv(valid_behaviors_path, BEHAVIOR_COLUMNS)

print(f"Loaded {len(train_news_df)} train news rows and {len(valid_news_df)} validation news rows from {LOCAL_DATA_ROOT}")
print(f"Loaded {len(train_behaviors_df)} train behaviors rows and {len(valid_behaviors_df)} validation behaviors rows")

Loaded 51282 train news rows and 42416 validation news rows from /home/asmvas/Documents/NTNU-prosjekt/anbefalingssytem/AnbSysProsjekt/smallDataset
Loaded 156965 train behaviors rows and 73152 validation behaviors rows


## Build ID mappings for ALS


In [5]:
all_news_ids = (
    pd.concat([train_news_df['nid'], valid_news_df['nid']], axis=0, ignore_index=True)
    .drop_duplicates()
    .tolist()
)
all_news_ids = [nid for nid in all_news_ids if isinstance(nid, str) and nid]
news_id_to_idx = {nid: idx for idx, nid in enumerate(all_news_ids)}
idx_to_news_id = np.array(all_news_ids)
NUM_ITEMS = len(all_news_ids)
print(f"Total unique news articles (train + validation): {NUM_ITEMS}")
NEGATIVE_SAMPLE_LIMIT = 50  # limit negatives per user to keep data tractable
RANDOM_SEED = 42

Total unique news articles (train + validation): 65238


## Construct training interactions for ALS


In [6]:
def build_user_feedback(behaviors_df: pd.DataFrame, news_index_map: Dict[str, int], negative_sample_limit: int | None = 50):
    user_feedback: Dict[str, Dict[int, float]] = defaultdict(dict)
    negative_counts = defaultdict(int)
    positive_events = 0
    negative_events = 0

    for row in behaviors_df.itertuples(index=False):
        user_id = getattr(row, 'user_id', None)
        impressions = getattr(row, 'impressions', '')
        history = getattr(row, 'history', '')
        if not isinstance(user_id, str) or not user_id:
            continue
        feedback = user_feedback[user_id]

        if isinstance(history, str) and history:
            for nid in history.split():
                item_idx = news_index_map.get(nid)
                if item_idx is None:
                    continue
                if feedback.get(item_idx, 0.0) < 1.0:
                    feedback[item_idx] = 1.0
                    positive_events += 1

        if isinstance(impressions, str) and impressions:
            for token in impressions.split():
                if '-' not in token:
                    continue
                nid, label = token.rsplit('-', 1)
                if not nid:
                    continue
                item_idx = news_index_map.get(nid)
                if item_idx is None:
                    continue
                is_click = 1.0 if label == '1' else 0.0
                if is_click == 1.0:
                    if feedback.get(item_idx, 0.0) < 1.0:
                        feedback[item_idx] = 1.0
                        positive_events += 1
                else:
                    if item_idx in feedback:
                        continue
                    if negative_sample_limit is not None and negative_counts[user_id] >= negative_sample_limit:
                        continue
                    feedback[item_idx] = 0.0
                    negative_counts[user_id] += 1
                    negative_events += 1

    return user_feedback, positive_events, negative_events

train_user_feedback, positive_count, negative_count = build_user_feedback(
    train_behaviors_df,
    news_id_to_idx,
    negative_sample_limit=NEGATIVE_SAMPLE_LIMIT,
)
print(f"Users with interactions: {len(train_user_feedback):,}")
print(f"Positive interactions kept: {positive_count:,}")
print(f"Sampled negative interactions kept: {negative_count:,}")

user_ids = sorted(train_user_feedback.keys())
user_id_to_idx = {uid: idx for idx, uid in enumerate(user_ids)}
NUM_USERS = len(user_ids)

user_item_indices: List[np.ndarray] = [np.empty((0,), dtype=np.int32) for _ in range(NUM_USERS)]
user_item_ratings: List[np.ndarray] = [np.empty((0,), dtype=np.float32) for _ in range(NUM_USERS)]
item_users: List[List[int]] = [[] for _ in range(NUM_ITEMS)]
item_ratings: List[List[float]] = [[] for _ in range(NUM_ITEMS)]
interaction_count = 0

for user_id, user_idx in user_id_to_idx.items():
    feedback = train_user_feedback[user_id]
    if feedback:
        item_idx = np.array(list(feedback.keys()), dtype=np.int32)
        rating_vals = np.array(list(feedback.values()), dtype=np.float32)
    else:
        item_idx = np.empty((0,), dtype=np.int32)
        rating_vals = np.empty((0,), dtype=np.float32)
    user_item_indices[user_idx] = item_idx
    user_item_ratings[user_idx] = rating_vals
    interaction_count += len(item_idx)
    for idx, rating in zip(item_idx, rating_vals):
        item_users[idx].append(user_idx)
        item_ratings[idx].append(float(rating))

item_users = [np.array(users, dtype=np.int32) if users else np.empty((0,), dtype=np.int32) for users in item_users]
item_ratings = [np.array(ratings, dtype=np.float32) if ratings else np.empty((0,), dtype=np.float32) for ratings in item_ratings]
active_user_indices = np.array([idx for idx, items in enumerate(user_item_indices) if items.size > 0], dtype=np.int32)
active_item_indices = np.array([idx for idx, users in enumerate(item_users) if users.size > 0], dtype=np.int32)

print(f"Total indexed interactions: {interaction_count:,}")
print(f"Active users: {len(active_user_indices):,} / {NUM_USERS:,}")
print(f"Active items: {len(active_item_indices):,} / {NUM_ITEMS:,}")

Users with interactions: 50,000
Positive interactions kept: 1,148,447
Sampled negative interactions kept: 1,847,091
Total indexed interactions: 2,984,572
Active users: 50,000 / 50,000
Active items: 47,496 / 65,238


## Train ALS recommender


In [7]:
def als_factorization(
    user_item_indices: List[np.ndarray],
    user_item_ratings: List[np.ndarray],
    item_users: List[np.ndarray],
    item_ratings: List[np.ndarray],
    active_user_indices: np.ndarray,
    active_item_indices: np.ndarray,
    latent_dim: int = 32,
    reg: float = 0.05,
    num_iters: int = 6,
    seed: int = 42,
):
    rng = np.random.default_rng(seed)
    user_factors = rng.normal(0, 0.01, size=(len(user_item_indices), latent_dim)).astype(np.float32)
    item_factors = rng.normal(0, 0.01, size=(len(item_users), latent_dim)).astype(np.float32)
    identity = np.eye(latent_dim, dtype=np.float32)

    for iteration in range(num_iters):
        for user_idx in active_user_indices:
            item_idx = user_item_indices[user_idx]
            ratings = user_item_ratings[user_idx]
            if item_idx.size == 0:
                continue
            item_vecs = item_factors[item_idx]
            A = item_vecs.T @ item_vecs + reg * identity
            b = item_vecs.T @ ratings
            user_factors[user_idx] = np.linalg.solve(A, b).astype(np.float32)

        for item_idx in active_item_indices:
            users = item_users[item_idx]
            ratings = item_ratings[item_idx]
            if users.size == 0:
                continue
            user_vecs = user_factors[users]
            A = user_vecs.T @ user_vecs + reg * identity
            b = user_vecs.T @ ratings
            item_factors[item_idx] = np.linalg.solve(A, b).astype(np.float32)

        print(f"Iteration {iteration + 1}/{num_iters} complete.")

    cold_item_mask = np.array([users.size == 0 for users in item_users])
    item_factors[cold_item_mask] = 0.0
    return user_factors, item_factors

ALS_FACTORS = 32
ALS_REG = 0.05
ALS_ITERS = 6
user_factors, item_factors = als_factorization(
    user_item_indices,
    user_item_ratings,
    item_users,
    item_ratings,
    active_user_indices,
    active_item_indices,
    latent_dim=ALS_FACTORS,
    reg=ALS_REG,
    num_iters=ALS_ITERS,
    seed=RANDOM_SEED,
)
print(f"User factors shape: {user_factors.shape}")
print(f"Item factors shape: {item_factors.shape}")

Iteration 1/6 complete.
Iteration 2/6 complete.
Iteration 3/6 complete.
Iteration 4/6 complete.
Iteration 5/6 complete.
Iteration 6/6 complete.
User factors shape: (50000, 32)
Item factors shape: (65238, 32)


## Evaluate on validation impressions


In [8]:
def history_to_indices(history_value: str) -> List[int]:
    if not isinstance(history_value, str) or not history_value:
        return []
    indices = []
    for nid in history_value.split():
        item_idx = news_id_to_idx.get(nid)
        if item_idx is not None:
            indices.append(item_idx)
    return indices


def build_user_vector(user_id: str, history_value: str, reg: float = 0.05) -> np.ndarray:
    base_vector = None
    interactions = dict(train_user_feedback.get(user_id, {}))
    train_idx = user_id_to_idx.get(user_id)
    if train_idx is not None and interactions:
        base_vector = user_factors[train_idx]

    for item_idx in history_to_indices(history_value):
        if interactions.get(item_idx, 0.0) < 1.0:
            interactions[item_idx] = 1.0

    if interactions:
        item_idx = np.array(list(interactions.keys()), dtype=np.int32)
        ratings = np.array(list(interactions.values()), dtype=np.float32)
        item_vecs = item_factors[item_idx]
        A = item_vecs.T @ item_vecs + reg * np.eye(item_vecs.shape[1], dtype=np.float32)
        b = item_vecs.T @ ratings
        return np.linalg.solve(A, b).astype(np.float32)

    if train_idx is not None:
        return user_factors[train_idx]

    return np.zeros(ALS_FACTORS, dtype=np.float32)


def dcg_at_k(sorted_labels: np.ndarray, k: int) -> float:
    if k <= 0:
        return 0.0
    labels_k = sorted_labels[:k]
    if labels_k.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, labels_k.size + 2))
    return float(np.sum(labels_k * discounts))


def ndcg_at_k(sorted_labels: np.ndarray, k: int) -> float:
    ideal = np.sort(sorted_labels)[::-1]
    ideal_dcg = dcg_at_k(ideal, k)
    if ideal_dcg == 0:
        return 0.0
    return dcg_at_k(sorted_labels, k) / ideal_dcg


def reciprocal_rank(sorted_labels: np.ndarray) -> float:
    positives = np.where(sorted_labels == 1)[0]
    if positives.size == 0:
        return 0.0
    return 1.0 / float(positives[0] + 1)


def binary_auc(scores: np.ndarray, labels: np.ndarray) -> float:
    pos_scores = scores[labels == 1]
    neg_scores = scores[labels == 0]
    if pos_scores.size == 0 or neg_scores.size == 0:
        return np.nan
    comparisons = (pos_scores[:, None] > neg_scores[None, :]).sum()
    ties = (pos_scores[:, None] == neg_scores[None, :]).sum()
    return float(comparisons + 0.5 * ties) / float(pos_scores.size * neg_scores.size)


metric_keys = ["ndcg@5", "ndcg@10", "hit@5", "hit@10", "mrr"]
metric_totals = {key: 0.0 for key in metric_keys}
auc_sum = 0.0
auc_count = 0
impressions_used = 0
skipped_impressions = 0

for row in valid_behaviors_df.itertuples(index=False):
    impressions_value = getattr(row, 'impressions', '')
    if not isinstance(impressions_value, str) or not impressions_value:
        skipped_impressions += 1
        continue

    candidate_indices = []
    labels = []
    for token in impressions_value.split():
        if '-' not in token:
            continue
        nid, label = token.rsplit('-', 1)
        item_idx = news_id_to_idx.get(nid)
        if item_idx is None:
            continue
        candidate_indices.append(item_idx)
        labels.append(int(label))

    if not candidate_indices:
        skipped_impressions += 1
        continue

    label_array = np.array(labels, dtype=np.int32)
    positives = int(label_array.sum())
    negatives = len(label_array) - positives
    if positives == 0 or negatives == 0:
        skipped_impressions += 1
        continue

    user_vector = build_user_vector(getattr(row, 'user_id', ''), getattr(row, 'history', ''), reg=ALS_REG)
    item_vecs = item_factors[candidate_indices]
    scores = item_vecs @ user_vector

    order = np.argsort(scores)[::-1]
    sorted_labels = label_array[order]

    metric_totals['hit@5'] += float(np.any(sorted_labels[:5]))
    metric_totals['hit@10'] += float(np.any(sorted_labels[:10]))
    metric_totals['ndcg@5'] += ndcg_at_k(sorted_labels, 5)
    metric_totals['ndcg@10'] += ndcg_at_k(sorted_labels, 10)
    metric_totals['mrr'] += reciprocal_rank(sorted_labels)

    auc_value = binary_auc(scores, label_array)
    if not np.isnan(auc_value):
        auc_sum += auc_value
        auc_count += 1

    impressions_used += 1

if impressions_used == 0:
    raise RuntimeError("No validation impressions with both positive and negative labels were found.")

evaluation_results = {key: metric_totals[key] / impressions_used for key in metric_keys}
evaluation_results['auc'] = auc_sum / auc_count if auc_count > 0 else float('nan')

print(f"Evaluated {impressions_used} validation impressions. Skipped {skipped_impressions} impressions.")
for key in metric_keys + ['auc']:
    value = evaluation_results[key]
    if np.isnan(value):
        print(f"{key}: NaN")
    else:
        print(f"{key}: {value:.4f}")

Evaluated 73152 validation impressions. Skipped 0 impressions.
ndcg@5: 0.2577
ndcg@10: 0.3168
hit@5: 0.4449
hit@10: 0.6277
mrr: 0.2771
auc: 0.5383
